In [17]:
from games.tictactoe.tictactoe import TicTacToe
from agents.agent_random import RandomAgent
from agents.minimax import MiniMax
from agents.minimax_alphabeta import MiniMaxAlphaBeta
import numpy as np
from tqdm import tqdm
from collections import defaultdict

In [2]:
game = TicTacToe(render_mode='')

In [3]:
game.agents

['X', 'O']

In [4]:
agents_rd = dict(map(lambda agent: (agent, RandomAgent(game=game, agent=agent)), game.agents))
agents_rd

{'X': <agents.agent_random.RandomAgent at 0x1a0ea9f8e50>,
 'O': <agents.agent_random.RandomAgent at 0x1a0ea9f9bd0>}

In [5]:
game.reset()
while not game.terminated():
    game.render()
    print(game.eval(game.agent_selection))
    action = agents_rd[game.agent_selection].action()
    game.step(action)
game.render()
print(game.eval(game.agent_selection))
print(game.rewards)

Player: X
Board:
 .  .  . 
 .  .  . 
 .  .  . 

0.0
Player: O
Board:
 X  .  . 
 .  .  . 
 .  .  . 

-0.375
Player: X
Board:
 X  .  . 
 .  .  . 
 O  .  . 

0.0
Player: O
Board:
 X  .  . 
 .  X  . 
 O  .  . 

-0.375
Player: X
Board:
 X  .  . 
 O  X  . 
 O  .  . 

0.25
Player: O
Board:
 X  .  X 
 O  X  . 
 O  .  . 

-0.375
Player: X
Board:
 X  O  X 
 O  X  . 
 O  .  . 

0.125
Player: O
Board:
 X  O  X 
 O  X  . 
 O  .  X 

-1
{'X': 1, 'O': -1}


In [6]:
players = {}
players[game.agents[0]] = MiniMax(game=game, agent=game.agents[0], depth=1)
players[game.agents[1]] = MiniMax(game=game, agent=game.agents[1], depth=4)


In [7]:
game.reset()
game.render()
print(game.observe(game.agents[0]))
action, value = players[game.agent_selection].minimax(game, depth=1)
print(action, value)
game.step(action)
game.render()
print(game.observe(game.agents[1]))
action, value = players[game.agent_selection].minimax(game, depth=4)
game.step(action)
print(action, value)
game.render()


Player: X
Board:
 .  .  . 
 .  .  . 
 .  .  . 

[[0 0 0]
 [0 0 0]
 [0 0 0]]
4 0.5
Player: O
Board:
 .  .  . 
 .  X  . 
 .  .  . 

[[0 0 0]
 [0 2 0]
 [0 0 0]]
3 -0.375
Player: X
Board:
 .  .  . 
 O  X  . 
 .  .  . 



In [8]:
values = defaultdict(list)
N = 10
for i in range(N):    
    game.reset()
    while not game.terminated():
        agent = game.agent_selection
        action = players[agent].action()
        game.step(action)
    for agent in game.agents:
        values[agent].append(game.reward(agent))
for agent in game.agents:
    print(f"Agent {agent} average reward: {np.mean(values[agent])} over {N} games")
    print(f"Agent {agent} rewards: {values[agent]}")

Agent X average reward: -0.3 over 10 games
Agent X rewards: [-1, -1, 1, 1, -1, -1, 1, -1, 0, -1]
Agent O average reward: 0.3 over 10 games
Agent O rewards: [1, 1, -1, -1, 1, 1, -1, 1, 0, 1]


## MiniMax con poda alfa-beta

Probaremos ahora nuestra propia imlpementación de MiniMax usando poda Alpha-Beta.

Esta implementación usa como base la provista en `minimax.py` y le agrega poda alpfa-beta y un contador de iteraciones `agent.iterations` (para comparar la efectividad de la poda).

In [ ]:
# Definimos dos agentes MiniMax, uno con poda alfa-beta y otro sin, ambos con misma profundidad
players = {}
players[game.agents[0]] = MiniMaxAlphaBeta(game=game, agent=game.agents[0], depth=5, alphabeta=False)
players[game.agents[1]] = MiniMaxAlphaBeta(game=game, agent=game.agents[1], depth=5, alphabeta=True)

In [29]:
game.reset()

player1 = players[game.agents[0]]
player2 = players[game.agents[1]]

action1 = player1.action()
action2 = player2.action()

print(f"Player 1 (AlphaBeta={player1.alphabeta}, Depth={player1.depth}) chose action: {action1} in {player1.iterations} iterations")
print(f"Player 2 (AlphaBeta={player2.alphabeta}, Depth={player2.depth}) chose action: {action2} in {player2.iterations} iterations")

Player 1 (AlphaBeta=False, Depth=5) chose action: 4 in 18730 iterations
Player 2 (AlphaBeta=True, Depth=5) chose action: 4 in 2110 iterations


Se aprecia que con poda de alfa-beta el cómputo de minimax en este juego se reduce considerablemente. Esto se debe a que existen muchas ramificaciones en las cuales un jugador Min sabe que Max no va a tomar ese camino porque existe otro mejor, y al revés respectivamente.

Esta poda nos permite implementar un grafo minimax completo (depth=9) sin evaluación. Sabemos que en el juego Ta-Te-Ti, si los jugadores juegan óptimo, siempre se termina en empate. Vamos a verificar esto corriendo varias iteraciones del juego con agentes con `depth=inf` y verificar que en todas las ocasiones se termine en empate.

Realizar esto sin usar poda alfa-beta sería extremadamente lento.

In [33]:
players = {}
players[game.agents[0]] = MiniMaxAlphaBeta(game=game, agent=game.agents[0], depth=float('inf'), alphabeta=True)
players[game.agents[1]] = MiniMaxAlphaBeta(game=game, agent=game.agents[1], depth=float('inf'), alphabeta=True)

values = defaultdict(list)
N = 10
for i in tqdm(range(N)):
    game.reset()
    while not game.terminated():
        agent = game.agent_selection
        action = players[agent].action()
        game.step(action)
    for agent in game.agents:
        values[agent].append(game.reward(agent))
for agent in game.agents:
    print(f"Agent {agent} average reward: {np.mean(values[agent])} over {N} games")
    print(f"Agent {agent} rewards: {values[agent]}")

100%|██████████| 10/10 [00:31<00:00,  3.13s/it]

Agent X average reward: 0.0 over 10 games
Agent X rewards: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Agent O average reward: 0.0 over 10 games
Agent O rewards: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
